In [ ]:
"""
依存:
  pip install opencv-python pillow pytesseract PyPDF2 tqdm
  ※ Tesseract本体のインストール必須:
    - Windows: https://github.com/UB-Mannheim/tesseract/wiki などから導入
    - macOS:   brew install tesseract
    - Linux:   sudo apt-get install tesseract-ocr tesseract-ocr-jpn

任意(幾何デワープ強化したい場合):
  git clone https://github.com/mzucker/page_dewarp
"""

from pathlib import Path
import tempfile
import subprocess
import shutil
import sys

import cv2
import numpy as np
from PIL import Image
import pytesseract
from PyPDF2 import PdfMerger
from tqdm import tqdm

# ========= 設定 =========
INPUT_DIR       = Path("./data/sample1/")                 # 画像群のフォルダ（ファイル名順に並べて処理）
OUTPUT_PDF      = Path("./output/book_ocr.pdf")    # 出力PDF
LANG            = "jpn+eng"                        # Tesseract言語。日本語中心なら "jpn" / 日英混在なら "jpn+eng"
DPI_FOR_PDF     = 300                              # PDF化の際の想定DPI（テキストマッピング精度が安定しやすい）

# (Windowsのみ) Tesseract本体のパスを直接指定したい場合はアンコメント
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# 幾何補正（デワープ）の利用可否
USE_PAGE_DEWARP = True                             # Trueなら mzucker/page_dewarp を外部呼び出し
PAGE_DEWARP_PY  = Path("./page_dewarp/page_dewarp.py")  # そのスクリプトへのパス

# 軽い見やすさ補正（幾何の後に適用）
APPLY_CLAHE     = True
APPLY_SAUVOLA   = False   # OCRはグレースケールでも良く読めるので通常は False。原稿が薄い/コントラスト乏しい場合だけ True

# ========= ここから実装 =========

def sort_images_by_name(input_dir: Path):
    exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
    files = [p for p in input_dir.iterdir() if p.suffix.lower() in exts]
    files.sort(key=lambda p: p.name)
    return files

def run_page_dewarp(img_path: Path, workdir: Path) -> Path:
    """
    mzucker/page_dewarp を外部プロセスとして呼ぶ。
    出力ファイル名は {stem}_dewarped.png になる前提（同リポの仕様）。
    """
    if not PAGE_DEWARP_PY.exists():
        raise FileNotFoundError(f"page_dewarp.py が見つかりません: {PAGE_DEWARP_PY}")

    # 一時ファイルのパスを作成（拡張子をpngに統一）
    tmp_in = workdir / f"{img_path.stem}_temp.png"
    
    # 入力画像をPNGとして保存（フォーマットを統一）
    img = imread_color(img_path)
    imwrite(tmp_in, img)
    
    if not tmp_in.exists():
        raise FileNotFoundError(f"一時ファイルの作成に失敗: {tmp_in}")

    # 実行
    cmd = [sys.executable, str(PAGE_DEWARP_PY), str(tmp_in)]
    try:
        result = subprocess.run(cmd, check=True, cwd=workdir, 
                              capture_output=True, text=True)
        if result.returncode != 0:
            print(f"page_dewarp出力: {result.stdout}\nエラー: {result.stderr}")
    except subprocess.CalledProcessError as e:
        print(f"page_dewarp実行エラー: {e}\n{e.stdout}\n{e.stderr}")
        raise

    out = workdir / f"{tmp_in.stem}_dewarped.png"
    if not out.exists():
        raise FileNotFoundError(f"page_dewarp 実行後の出力が見つかりません: {out}")
    
    return out

def enhance_image(img_bgr: np.ndarray) -> np.ndarray:
    """
    読みやすさ向上の軽い処理（幾何補正後に適用）
    - CLAHEで輝度だけ局所コントラスト強調
    - 必要に応じてSauvolaで二値化（OCRのために必須ではない）
    """
    out = img_bgr.copy()

    if APPLY_CLAHE:
        lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l2 = clahe.apply(l)
        out = cv2.cvtColor(cv2.merge([l2, a, b]), cv2.COLOR_LAB2BGR)

    if APPLY_SAUVOLA:
        gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
        # Sauvola系の局所二値化（ximgproc の niBlackThreshold を利用）
        th = cv2.ximgproc.niBlackThreshold(gray, 255, cv2.THRESH_BINARY,
                                           blockSize=51, k=-0.2)
        out = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

    return out

def imread_color(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)  # 日本語パス対応
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"画像が読み込めません: {path}")
    return img

def imwrite(path: Path, img_bgr: np.ndarray):
    path.parent.mkdir(parents=True, exist_ok=True)
    ext = path.suffix.lower()
    params = []
    if ext in [".jpg", ".jpeg"]:
        params = [cv2.IMWRITE_JPEG_QUALITY, 95]
    data = cv2.imencode(ext if ext else ".png", img_bgr, params)[1]
    path.write_bytes(data.tobytes())

def ocr_image_to_pdf_bytes(img_bgr: np.ndarray, lang: str, dpi: int) -> bytes:
    """
    Tesseractで“画像+不可視テキスト層”のPDFを生成してbytesで返す。
    """
    # PIL Imageに変換し、解像度情報（dpi）を持たせる
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    pdf_bytes = pytesseract.image_to_pdf_or_hocr(
        pil, lang=lang, extension="pdf", config=f"--dpi {dpi}"
    )
    return pdf_bytes

def main():
    images = sort_images_by_name(INPUT_DIR)
    if not images:
        print(f"画像が見つかりません: {INPUT_DIR}")
        return

    OUTPUT_PDF.parent.mkdir(parents=True, exist_ok=True)
    
    # 一時PDFファイルのパスを保持するリスト
    pdf_files = []
    
    with tempfile.TemporaryDirectory() as td:
        tmpdir = Path(td)
        print(f"一時ディレクトリ: {tmpdir}")

        for img_path in tqdm(images, desc="Processing"):
            # 1) デワープ（任意）
            try:
                if USE_PAGE_DEWARP:
                    dewarped_path = run_page_dewarp(img_path, tmpdir)
                    img = imread_color(dewarped_path)
                else:
                    img = imread_color(img_path)  # そのまま使う
            except Exception as e:
                # デワープに失敗したら元画像で継続
                print(f"[WARN] dewarp失敗 ({img_path.name}): {e}")
                img = imread_color(img_path)

            # 2) 軽い画質補正
            img = enhance_image(img)

            # 3) OCR → 単ページPDF化
            pdf_bytes = ocr_image_to_pdf_bytes(img, LANG, DPI_FOR_PDF)

            # 4) 一旦一時PDFとして保存
            page_pdf = tmpdir / f"{img_path.stem}.pdf"
            page_pdf.write_bytes(pdf_bytes)
            pdf_files.append(str(page_pdf))

        # 5) 全ページをマージ
        merger = PdfMerger()
        for pdf_file in pdf_files:
            merger.append(pdf_file)
        
        # 6) 1冊のPDFとして保存
        with open(OUTPUT_PDF, "wb") as f:
            merger.write(f)

    print(f"Done. Saved: {OUTPUT_PDF.resolve()}")

if __name__ == "__main__":
    main()
